# 5-1 Assignment: Cartpole Problem
___
<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Run all code blocks and observe the output. Once you have reviewed the sample output. Use the <b>LastName_FirstName_Assignment2.ipynb</b> file to complete your assignment.
</div>

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> For compatability purposes, libraries have been updated from those used in the required readings to match to current versions; hence some of the package invocations may differ slightly from the book. The affected lines of code have comments added to the right as applicable, with the old code commented out above for reference.
</div>

<div class="alert alert-block alert-danger" style="color:black;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
<b>Example messages:</b>
    <ul>
        <li>Could not find cuda drivers on your machine, GPU will not be used.</li>
        <li>Please check linkage and avoid linking the same target more than once.</li>
        <li>E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)</li>
        <li>Allocation of ######## exceeds 10% of free system memory</li>
    </ul>
</div>

---

### Installing Required Packages
This is to install necessary components to run the assignment

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> If the assignment takes too long, you may want to download the code and run it locally to take advantage of your GPU.
</div>

In [2]:
# Package Imports
import numpy as np
import tensorflow as tf
import gymnasium as gym
from collections import deque
import random

# Environment setup and Variables
try:
    env = gym.make("CartPole-v1", render_mode=None)
except Exception:
    print('Failed to initialize environment! Make sure that gymnasium was installed correctly!')
    sys.exit(1)

state_shape = env.observation_space.shape[0]
action_size = env.action_space.n

# Hyperparameters
GAMMA = 0.99
EXPLORATION_MAX = 1.0
EXPLORATION_MIN = 0.01
EXPLORATION_DECAY = 0.990
LEARNING_RATE = 0.001
BATCH_SIZE = 64
TRAIN_START = 1000
MEMORY_SIZE = 2000
EPISODES = 300

# Counters during training
train_freq = 4
step_count = 0
target_update_freq = 10
# If an error occurs below, it is because the environment is looking for a GPU.

In [3]:
memory = deque(maxlen=MEMORY_SIZE)

# DQN Builder
def build_dqn_model():
    try:
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(state_shape,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(action_size, activation='linear')
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
        return model
    except Exception:
        print('Error occurred while building the DQN model!')
        sys.exit(1)

In [4]:
# Calling the build_dqn_model method
model = build_dqn_model()
target_model = build_dqn_model()
target_model.set_weights(model.get_weights())

I0000 00:00:1775184142.768996    8878 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1775184142.842746    8878 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1775184142.844803    8878 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1775184142.851579    8878 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> If an error displays, it is simply trying to connect to the GPU.
</div>


In [5]:
# Random Action selection for exploration
def get_action(state, EXPLORATION_MAX):
    try:
        if np.random.rand() <= EXPLORATION_MAX:
            return random.randrange(action_size)
        q_values = model.predict(np.array([state]), verbose=0) #Remove verbose=0 to see every prediction display
        return np.argmax(q_values[0])
    except Exception:
            print("Error during get_action method!")
            sys.exit(1)

In [6]:
# Experience Replay
def experience_replay():
    if len(memory) < TRAIN_START:
        return

    batch = random.sample(memory, BATCH_SIZE)
    states = np.zeros((BATCH_SIZE, state_shape))
    next_states = np.zeros((BATCH_SIZE, state_shape))
    actions, rewards, completions = [], [], []

    try:
        for i, (state, action, reward, state_next, terminal) in enumerate(batch):
            states[i] = state
            next_states[i] = state_next
            actions.append(action)
            rewards.append(reward)
            completions.append(terminal)

        target_q = model.predict(states, verbose=0) #Remove verbose=0 to see every prediction display
        next_q = target_model.predict(next_states, verbose=0) #Remove verbose=0 to see every prediction display

        for i in range(BATCH_SIZE):
            if completions[i]:
                target_q[i][actions[i]] = rewards[i]
            else:
                target_q[i][actions[i]] = rewards[i] + GAMMA * np.max(next_q[i])

        model.fit(states, target_q, batch_size=BATCH_SIZE, verbose=0)
    except Exception:
        print("Error during replay")
        sys.exit(1)

In [7]:
# A list to keep track of all of the episodes
episode_rewards = []

# Main loop
# Will iterate through based on the number of EPISODES originally listed up top
for e in range(EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = get_action(state, EXPLORATION_MAX)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        reward = reward if not done else -10

        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward
        
        step_count += 1
        if step_count % train_freq == 0:
            experience_replay()

    if EXPLORATION_MAX > EXPLORATION_MIN:
        EXPLORATION_MAX *= EXPLORATION_DECAY
        EXPLORATION_MAX = max(EXPLORATION_MIN, EXPLORATION_MAX)

    if e % target_update_freq == 0:
        target_model.set_weights(model.get_weights())

    episode_rewards.append(total_reward)
    # This will display the average reward of every 10 episodes occurring.
    # Comment this section and add the below print method if you want to display every episode
    if e % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {e+1}: Average Reward = {avg_reward:.2f}, Exploration = {EXPLORATION_MAX:.3f}")
    
    # Logic to complete execution if average is greater than 195
    if len(episode_rewards) >= 100:
        avg_last_hun = np.mean(episode_rewards[-100:])
        if avg_last_hun >= 195:
            print(f"Solved at episode {e}: Average reward over last 100: {avg_last_hun:.2f}")
            break
        
    # Comment this in if you want to see every iteration of the training occurring     
    # print(f"Episode {e+1}: Total Reward = {total_reward}, Exploration = {EXPLORATION_MAX:.3f}")

# Saving model as needed    
model.save("Cartpole_model.h5")
print("Training complete! Model saved as Cartpole_model")

Episode 1: Average Reward = 13.00, Exploration = 0.990


I0000 00:00:1775184143.971178    8937 service.cc:146] XLA service 0x78661c003810 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775184143.971232    8937 service.cc:154]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775184144.172448    8937 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Episode 11: Average Reward = 15.70, Exploration = 0.895
Episode 21: Average Reward = 16.10, Exploration = 0.810
Episode 31: Average Reward = 16.90, Exploration = 0.732
Episode 41: Average Reward = 14.30, Exploration = 0.662
Episode 51: Average Reward = 5.30, Exploration = 0.599
Episode 61: Average Reward = 1.80, Exploration = 0.542
Episode 71: Average Reward = 4.90, Exploration = 0.490
Episode 81: Average Reward = 13.30, Exploration = 0.443
Episode 91: Average Reward = 23.60, Exploration = 0.401
Episode 101: Average Reward = 45.10, Exploration = 0.362
Episode 111: Average Reward = 170.00, Exploration = 0.328
Episode 121: Average Reward = 233.20, Exploration = 0.296
Episode 131: Average Reward = 171.30, Exploration = 0.268
Episode 141: Average Reward = 231.80, Exploration = 0.242
Episode 151: Average Reward = 254.90, Exploration = 0.219
Episode 161: Average Reward = 284.20, Exploration = 0.198
Episode 171: Average Reward = 200.00, Exploration = 0.179
Episode 181: Average Reward = 296.00

Solved at episode 183: Average reward over last 100: 197.74
Training complete! Model saved as Cartpole_model


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b><br>
    If an error displays, it is attempting to connect to the GPU.<br>
    It will not connect and run on CPU. <br>
    Code will take some time to run, which is commonplace for real life models.<br>
    <b><i>-- Make sure your computer does not go to sleep, and take a well-deserved break! --</b></i>
</div>


## Analysis of the Cartpole Problem

The cartpole problem is a good example of how reinforcement learning works in practice because the agent is not told the correct move ahead of time. Instead, it learns by trying actions, seeing what happens, and getting rewarded for staying balanced longer. In this setup, the main goal of the agent is to keep the pole upright while also keeping the cart within the allowed range of motion. The longer it can do that, the better the reward it earns. Over time, the agent starts to connect certain states with actions that lead to better results.

The state of the environment is made up of four values: cart position, cart velocity, pole angle, and pole angular velocity. These values describe what is happening at a specific moment and give the agent the information it needs to make a decision. The action choices are simple because the agent can only move the cart in one of two directions: left or right. Even with only two actions, the problem is still challenging because the agent has to react correctly to constant changes in balance and motion.

The algorithm used for this problem is deep Q-learning, also called a Deep Q-Network or DQN. Regular Q-learning depends on a Q-table that stores a value for every state-action pair. That works for small, simple problems, but cartpole has continuous state values, so a normal Q-table would not be practical. Deep Q-learning handles this by using a neural network to estimate the Q-values instead of storing every possibility directly. That makes it much more realistic for a problem like this.

Experience replay is one of the parts of the algorithm that helps make learning more stable. As the agent moves through episodes, it stores past experiences in memory. Each stored experience includes the current state, the action chosen, the reward, the next state, and whether the episode ended. During training, the model pulls random samples from that memory instead of learning only from the newest step. This matters because consecutive experiences are often very similar, and training on only the latest moves can make learning unstable. By mixing older and newer experiences together, the network gets a better spread of examples and can learn in a more balanced way.

The discount factor is also important because it controls how much the model values future rewards. If the discount factor is high, the agent puts more emphasis on long-term success. In the cartpole problem, that usually helps because the best move is the one that keeps the pole balanced for future time steps, not just the move that looks safest right now. If the discount factor is too low, the agent focuses more on immediate reward and may not learn as strong of a long-term balancing strategy. Because of that, the discount factor plays a big role in how far ahead the model is effectively “thinking.”

Neural networks are what make deep Q-learning possible in this assignment. The network takes in the four state values as inputs and outputs Q-values for the two possible actions. In other words, it estimates how good moving left or right would be from the current state. The hidden layers allow the model to learn patterns between the state inputs and the expected rewards. This is more efficient than standard Q-learning because the model can generalize from similar situations instead of treating every state like a completely separate case. That makes a big difference in a problem where the state values are constantly changing.

From the training results, it was clear that the agent improved as the episodes continued and exploration gradually decreased. Early on, the rewards were much lower because the agent was still trying random actions and learning how the environment responded. Later in training, the average reward increased enough for the environment to be considered solved, which showed that the model had learned a much more consistent balancing strategy. The run solved the environment at episode 183 with an average reward over the last 100 episodes of 197.74, which shows strong overall improvement.

Changing the learning rate also affected performance. When the learning rate is set higher, the model updates more aggressively, so it can improve faster in the beginning. The downside is that it can also become less stable because larger updates may cause it to jump past better solutions. When the learning rate is lower, the model usually learns more steadily, but it takes longer to improve because each update is smaller. Based on that, the learning rate has a direct effect on both speed and stability. A higher value can help the model learn faster, but a lower one often gives smoother progress.

Overall, the cartpole assignment shows how several reinforcement learning ideas work together. The agent learns from rewards, experience replay makes the training process more reliable, the discount factor helps it value future rewards, and the neural network allows the model to handle a continuous state space. Once those pieces start working together, the agent moves from random guessing to making actions that keep the pole balanced much more effectively.

## References

Brownlee, J. (2017, July 11). *A gentle introduction to Q-learning*. Machine Learning Mastery. https://machinelearningmastery.com/what-is-reinforcement-learning/

FreeCodeCamp. (n.d.). *An introduction to Q-learning: Reinforcement learning*. https://medium.com/free-code-camp/an-introduction-to-q-learning-reinforcement-learning-14ac0b4493cc

Serengeti Tech. (n.d.). *Using Q-learning for pathfinding*. https://serengetitech.com/tech/using-q-learning-for-pathfinding/

Violante, A. (2019, March 18). *Simple reinforcement learning: Q-learning*. Towards Data Science. https://towardsdatascience.com/simple-reinforcement-learning-q-learning-fcddc4b6fe56